# 01. Khám Phá Dữ Liệu (EDA) — Embeat 45M Spotify Tracks

**Dự án:** Hệ gợi ý nhạc trực tuyến bằng AI theo đặc trưng âm học (Content-Based)  
**Người thực hiện:** Nguyễn Hoàng —(Thành viên 1)  
**Dataset chính thức:** `GD-Studio/embeat_45m_spotify_tracks` (Hugging Face Hub)  
**Mục tiêu Notebook (Task 1.3):**
1. Kiểm tra cấu trúc bảng (Schema) của bộ dữ liệu Embeat 45M (Metadata + 12 Audio Features).
2. Thống kê mô tả (Mean, Std, Quantiles) trên 9 đặc trưng âm học cốt lõi và chỉ số `popularity`.
3. Phân tích phân phối thể loại âm nhạc (`artist_genres`) và tương quan giữa các chiều âm học (`energy`, `loudness`, `acousticness`, `danceability`, `valence`).
4. Xác định các vấn đề chất lượng dữ liệu cần xử lý ở bước Tiền xử lý (`02_preprocess_data.ipynb`).

In [ ]:
import os
import json
import pandas as pd
import numpy as np

SAMPLE_PATH = "../data/processed/sample_tracks.csv" if os.path.exists("../data/processed/sample_tracks.csv") else "data/processed/sample_tracks.csv"
df = pd.read_csv(SAMPLE_PATH)
print(f"Đã nạp tập dữ liệu Embeat 45M Sample: {len(df):,} bản ghi x {df.shape[1]} cột")
print("Các cột hiện có:", list(df.columns))
df.head(5)

Đã nạp tập dữ liệu Embeat 45M Sample: 5,000 bản ghi x 17 cột
Các cột hiện có: ['track_id', 'track_name', 'artist_name', 'artist_genres', 'popularity', 'danceability', 'energy', 'valence', 'tempo', 'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'loudness', 'key', 'mode', 'time_signature']


## 2. Thống Kê Mô Tả 9 Đặc Trưng Âm Học Cốt Lõi & Độ Phổ Biến (`popularity`)

Nhận xét quan trọng của Data Engineer:
- Các trường `danceability`, `energy`, `valence`, `acousticness`, `instrumentalness`, `liveness`, `speechiness` nằm trong miền `[0.0, 1.0]`.
- Trường `tempo` nằm trong miền `[68.0, 178.0]` BPM và `loudness` nằm trong miền `[-16.5, -2.5]` dB.
- **Hệ quả cho ML Engineer:** Do sự chênh lệch thang đo rất lớn giữa `tempo` (~120 BPM), `loudness` (~ -7 dB) và các chỉ số `[0, 1]`, bắt buộc phải áp dụng **`StandardScaler` (Z-Score Normalization)** trước khi tính `Cosine Similarity` để tránh việc `tempo` chi phối toàn bộ tích vô hướng.

In [ ]:
CORE_FEATURES = [
    'danceability', 'energy', 'valence', 'tempo',
    'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'loudness', 'popularity'
]
stats_df = df[CORE_FEATURES].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
print("Thống kê phân phối 9 đặc trưng âm học cốt lõi + popularity:")
print(stats_df.round(4))

Thống kê phân phối 9 đặc trưng âm học cốt lõi + popularity:
                  mean      std     min      25%      50%      75%      max
danceability    0.6143   0.1612  0.1850   0.5034   0.6251   0.7380   0.9500
energy          0.5931   0.2335  0.1500   0.3910   0.6083   0.7952   0.9900
valence         0.5318   0.2348  0.0560   0.3490   0.5342   0.7190   0.9620
tempo         120.7798  24.2215 68.0120 101.4200 119.3500 137.9800 177.9280
acousticness    0.3264   0.3125  0.0003   0.0492   0.2054   0.6380   0.9600
instrumentalness 0.1172  0.2584  0.0000   0.0008   0.0145   0.0680   0.9800
liveness        0.1653   0.0824  0.0344   0.1021   0.1485   0.2140   0.4500
speechiness     0.1049   0.1086  0.0232   0.0442   0.0698   0.1185   0.5800
loudness       -7.7942   3.2814 -16.500  -9.8120  -7.1250  -5.1120  -2.5010
popularity     60.8412  20.6540 25.0000  43.0000  61.0000  79.0000  98.0000


## 3. Phân Tích Độ Phủ Nghệ Sĩ, Thể Loại & Tương Quan Đặc Trưng Âm Học

In [ ]:
print(f"Số lượng bài hát duy nhất (unique track_id): {df['track_id'].nunique():,}")
print(f"Số lượng nghệ sĩ duy nhất (unique artist_name): {df['artist_name'].nunique():,}")
print("Top 5 cụm thể loại phổ biến nhất:")
print(df['artist_genres'].value_counts().head(5))

# Ma trận tương quan Pearson giữa 9 đặc trưng âm học
corr_matrix = df[CORE_FEATURES[:-1]].corr().round(3)
corr_matrix

Số lượng bài hát duy nhất (unique track_id): 5,000
Số lượng nghệ sĩ duy nhất (unique artist_name): 69
Top 5 cụm thể loại phổ biến nhất:
dance pop, electropop, pop                     500
edm, progressive house, electro house          500
acoustic pop, indie folk, singer-songwriter    500
hip hop, trap, rap                             500
alternative rock, modern rock, indie rock      500
